<a href="https://colab.research.google.com/github/manikantavs01/DeepLearning_Hackaton/blob/main/genai_support_ticket_classification_fewshot_groq.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
#importing libraries
import os
import pandas as pd
import ast
import time

In [12]:
from getpass import getpass

key = getpass('Please enter your together AI API Key here: ')

Please enter your together AI API Key here: ··········


In [14]:
os.environ['GROQ_API_KEY'] = key

In [ ]:
#installing groq
!pip install groq

In [15]:
#setting client and model
from groq import Groq
client=Groq()
model="meta-llama/Llama-3.3-70B-Instruct-Turbo"

In [16]:
# Reding test and train data
test = pd.read_csv('test_genai.csv')
train = pd.read_csv('train_genai.csv')

In [17]:
#chat function
def get_response(prompt, model=model):
    messages = [{"role":"user","content":prompt}]
    client = Groq()
    response = client.chat.completions.create(model=model,messages=messages)
    return response.choices[0].message.content

In [18]:
#Few shot examples
few_shot_examples = f''' classify the customer support tickets in english such as
  department as one of Technical Support, Customer Service, Billing and Payments, Product Support, IT Support, Returns and Exchanges, Sales and Pre-Sales, Human Resources, Service Outages and Maintenance, General Inquiry
  type as one of Incident, Request, Change, problem
  priority as low, medium, high
  language as the language code for the email's language
  without any reasoning strictly in dictionary format without any prefixes'''

# Add the 8 labeled training emails as few-shot examples
for _, row in train.iterrows():
    few_shot_examples += f'Ticket: "{row["ticket_body"]}"\n' \
                       f'Department: {row["department"]}\n' \
                       f'Type: {row["type"]}\n' \
                       f'Priority: {row["priority"]}\n' \
                       f'Language: {row["language"]}\n\n'

In [27]:
few_shot_examples+test["ticket_body"][0]

' classify the customer support tickets in english such as\n  department as one of Technical Support, Customer Service, Billing and Payments, Product Support, IT Support, Returns and Exchanges, Sales and Pre-Sales, Human Resources, Service Outages and Maintenance, General Inquiry\n  type as one of Incident, Request, Change, problem\n  priority as low, medium, high\n  language as the language code for the email\'s language\n  without any reasoning strictly in dictionary format without any prefixesTicket: "Estimado equipo de soporte de TI,\n\nEstoy escribiendo para informar un monto de facturación incorrecto en mi suscripción de Google Workspace Business Standard bajo la cuenta <acc_num>. Por favor, revise y ajuste la factura. Espero su pronta respuesta.\n\nSaludos,\n\n<name>"\nDepartment: Billing and Payments\nType: Incident\nPriority: low\nLanguage: es\n\nTicket: "Dear IT Services Support Team, I hope this message finds you well. My name is <name> and I am currently experiencing critic

In [19]:
# Function to classify test emails in batches
def classify_tickets(tickets):
    classified_output = []
    for email in tickets:
      input_prompt = few_shot_examples + f'Ticket: "{email}"\n' \
                                        f'Department:\nType:\nPriority:\nLanguage:'
     # print(input_prompt)
      try:
        response = get_response(input_prompt)
        output = response.strip().split("\n")
        classified_output.append({
              "email": email,
              "department": output[0].replace("Department:", "").strip() if len(output) > 0 else "Unknown",
              "type": output[1].replace("Type:", "").strip() if len(output) > 1 else "Unknown",
              "priority": output[2].replace("Priority:", "").strip() if len(output) > 2 else "Unknown",
              "language": output[3].replace("Language:", "").strip() if len(output) > 3 else "Unknown"
              })
      except Exception as e:
        classified_output.append({
            "email": email,
            "department": "Error",
            "type": "Error",
            "priority": "Error",
            "language": "Error"
            })
    return(classified_output)

In [28]:
get_response(few_shot_examples+test["ticket_body"][0])

NotFoundError: Error code: 404 - {'error': {'message': 'The model `meta-llama/llama-3.3-70b-instruct-turbo` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'code': 'model_not_found'}}

In [23]:
# Run classification on the test emails
test_emails = train["ticket_body"].tolist()
classified_data = classify_tickets(test_emails)


In [24]:
classified_data=pd.DataFrame(classified_data)
classified_data.to_csv("classified_tickets.csv", index=False)
print("Classification completed. Results saved to 'classified_tickets.csv'.")

Classification completed. Results saved to 'classified_tickets.csv'.


In [25]:
classified_data.head()

,email,department,type,priority,language
0,"Estimado equipo de soporte de TI,\n\nEstoy esc...",Error,Error,Error,Error
1,"Dear IT Services Support Team, I hope this mes...",Error,Error,Error,Error
2,"Estimado Servicio de Atención al Cliente,\n\nM...",Error,Error,Error,Error
3,"Cher service client, \n\nJe vous écris pour de...",Error,Error,Error,Error
4,"Dear Customer Support Team,\n\nI am encounteri...",Error,Error,Error,Error


In [ ]:
classified_data.to_csv("classified_tickets.csv", index=False)
print("Classification completed. Results saved to 'classified_tickets.csv'.")

Classification completed. Results saved to 'classified_tickets.csv'.
